# Curriculum 01 · Lab 1 — Fixed vs Recursive

**Goal:** Run the two classic text-splitting strategies head to head on the
same three public-domain Project Gutenberg novels and see where each one's cuts
land.

```
Splitter (fixed) : CharacterTextSplitter(separator="")   — pure character counter
Splitter (rec)   : DocumentProcessor → RecursiveCharacterTextSplitter — separator ladder
Data             : Data/corpus/gutenberg/ (pride-and-prejudice, moby-dick, a-tale-of-two-cities)
Loader           : GutenbergLoader (strips the Gutenberg license preamble/footer)
Budget           : chunk_size=500, chunk_overlap=50
```

The first decision in every RAG pipeline is: *how do I cut a long document into
chunks?* This lab answers it by comparing the two classic answers on identical
input and identical budget — and then inspecting *where* the cuts land, because
that is what determines whether embeddings see clean text at every chunk
boundary.


## 0 · Setup — imports & optional installs

Two things to know before running:

* **`langchain-core` / `langchain-text-splitters`** provide `Document`,
  `CharacterTextSplitter` and (via `src/splitters/recursive.py`)
  `RecursiveCharacterTextSplitter` — install cell below if missing.
* **Repo-root component library.** The lab imports
  `src/loaders/gutenberg.GutenbergLoader` and `src/splitters/recursive.DocumentProcessor`
  from the repo root. The `.py` resolves that root with
  `Path(__file__).resolve().parents[2]`; a notebook has no `__file__`, so the
  cell below resolves the repo root from the working directory instead — it
  works when run from the repo root, and walks up to find the folder containing
  `src/splitters/` if you run from the notebook's own folder.
* **Kernel cwd.** Jupyter launches kernels with the *notebook's* directory as
  the working directory (not the terminal's), so the data paths in the
  configuration cell are anchored to `REPO_ROOT` — the same repo-root-relative
  semantics the `.py` uses.


In [1]:
# Needed for THIS lab only (already in requirements.txt):
#   langchain-core            → Document
#   langchain-text-splitters  → CharacterTextSplitter / RecursiveCharacterTextSplitter
%pip install langchain-core langchain-text-splitters



[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
import sys
from pathlib import Path

from langchain_core.documents import Document
from langchain_text_splitters import CharacterTextSplitter

# Make the repo-root component library importable. The .py uses
# ``Path(__file__).resolve().parents[2]``; a notebook has no ``__file__``, so
# resolve the repo root from the working directory instead (works when run
# from the repo root, and walks up if run from the notebook's own folder).
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "src" / "splitters").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from loaders.gutenberg import GutenbergLoader  # noqa: E402
from splitters.recursive import DocumentProcessor  # noqa: E402


## 1 · Configuration — tweak these to rerun the comparison

Everything the lab measures is driven by these four knobs. `CHUNK_SIZE` is the
target length of every chunk, `CHUNK_OVERLAP` is how many characters bleed into
the next chunk (so context is never lost exactly at a boundary), and `PREVIEW`
controls how much text is shown on each side of a cut in the boundary analysis.
`DOC_PATHS` is the corpus: three public-domain Gutenberg novels, loaded through
`GutenbergLoader` which strips the license preamble/footer.


In [3]:
# Data paths are repo-root-relative in the .py; anchor them to REPO_ROOT so the
# notebook works no matter which directory the kernel starts in (jupyter
# launches kernels with the notebook's directory as cwd).
DATA_ROOT = REPO_ROOT / "Data"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
PREVIEW = 100  # max characters shown on each side of a cut
DOC_PATHS = [
    DATA_ROOT / "corpus/gutenberg/pride-and-prejudice.txt",
    DATA_ROOT / "corpus/gutenberg/moby-dick.txt",
    DATA_ROOT / "corpus/gutenberg/a-tale-of-two-cities.txt",
]


## 2 · Load — Gutenberg books via the shared loader

`GutenbergLoader` is the repo's loader block for Project Gutenberg plain text:
it strips everything between the `*** START OF THE PROJECT GUTENBERG EBOOK` and
`*** END OF THE PROJECT GUTENBERG EBOOK` markers and sets `metadata["source"]`
to the book path. That boilerplate is noise for chunking — the license preamble
would otherwise be split and embedded like real content. Each book becomes one
`Document`; the three books are concatenated into a single list.


In [4]:
def load_docs(paths: list[Path]) -> list[Document]:
    """Load each book through ``GutenbergLoader`` (strips the Gutenberg
    license preamble/footer, sets ``metadata["source"]`` to the book path)."""
    docs = []
    for path in paths:
        docs.extend(GutenbergLoader(path).load())
    return docs


# --- 2. Load ---------------------------------------------------------
docs = load_docs(DOC_PATHS)
print("=" * 66)
print("Lab 01 — fixed vs recursive text splitting")
print(f"chunk_size={CHUNK_SIZE}, chunk_overlap={CHUNK_OVERLAP}")
print("=" * 66)
print(f"\n[1] Loaded {len(docs)} document(s):")
for doc in docs:
    print(f"    {Path(doc.metadata['source']).name:<22} {len(doc.page_content):>5} chars")


Lab 01 — fixed vs recursive text splitting
chunk_size=500, chunk_overlap=50

[1] Loaded 3 document(s):
    pride-and-prejudice.txt 728769 chars
    moby-dick.txt          1218971 chars
    a-tale-of-two-cities.txt 757631 chars


## 3 · Split — fixed vs recursive splitters

The two strategies, both given the same `chunk_size=500, chunk_overlap=50`:

* **FIXED** — `CharacterTextSplitter(separator="")` is a pure character counter:
  it cuts at exactly `chunk_size` characters, no matter where that lands. Words
  and sentences get torn in half; the chunk count is purely a function of text
  length. (Note: `CharacterTextSplitter` defaults to a `\n\n` separator, which
  would quietly respect paragraphs — passing `separator=""` turns it into the
  blind character counter that "fixed splitting" really means.)
* **RECURSIVE** — `DocumentProcessor` wraps `RecursiveCharacterTextSplitter`,
  which climbs a ladder of separators — paragraphs (`\n\n`), newlines (`\n`),
  spaces — and only falls back to characters when nothing else fits. Chunks end
  on paragraph/word boundaries instead of mid-word.

Why it matters: embeddings are trained on whole words and sentences. A chunk
that starts or ends mid-word feeds garbage tokens at exactly the boundary points
where neighbouring chunks meet, so retrieval quality degrades where structure
matters most.


In [5]:
# --- 3. Split --------------------------------------------------------
fixed_chunks = CharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, separator=""
).split_documents(docs)
recursive_chunks = DocumentProcessor(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
).split_docs(docs)
print("\n[2] Split with both splitters (same chunk_size/chunk_overlap):")
print(f"    CharacterTextSplitter(separator='') : {len(fixed_chunks):>2} chunks")
print(f"    DocumentProcessor (recursive)        : {len(recursive_chunks):>2} chunks")



[2] Split with both splitters (same chunk_size/chunk_overlap):
    CharacterTextSplitter(separator='') : 6013 chunks
    DocumentProcessor (recursive)        : 7396 chunks


## 4 · Compare — chunk length stats

`chunk_length_stats` reduces each splitter's output to four numbers: chunk
count, average length, and the min/max. Fixed splitting produces a predictable
distribution (every chunk is ~`chunk_size` long, so the count is roughly
`total_chars / chunk_size`). Recursive splitting produces fewer, longer chunks
because it refuses to cut mid-paragraph — the count is a function of the
document's *structure*, not just its length.


In [6]:
def chunk_length_stats(chunks: list[Document]) -> tuple[int, float, int, int]:
    """Return (count, avg, min, max) chunk length in characters."""
    lengths = [len(c.page_content) for c in chunks]
    count = len(lengths)
    if count == 0:
        return 0, 0.0, 0, 0
    return count, sum(lengths) / count, min(lengths), max(lengths)


# --- 4. Compare lengths ----------------------------------------------
f_count, f_avg, f_min, f_max = chunk_length_stats(fixed_chunks)
r_count, r_avg, r_min, r_max = chunk_length_stats(recursive_chunks)
print("\n[3] Chunk length stats (characters):")
print(f"    {'':<12}{'chunks':>7}{'avg':>9}{'min':>7}{'max':>7}")
print(f"    {'fixed':<12}{f_count:>7d}{f_avg:>9.1f}{f_min:>7d}{f_max:>7d}")
print(f"    {'recursive':<12}{r_count:>7d}{r_avg:>9.1f}{r_min:>7d}{r_max:>7d}")



[3] Chunk length stats (characters):
                 chunks      avg    min    max
    fixed          6013    499.4    194    500
    recursive      7396    365.4      5    499


## 5 · Boundary quality — where do the cuts land?

Counting chunks only tells you *how many*; the interesting question is *where*.
These helpers inspect the actual cut points:

* `cut_stats` counts in-document cuts and how many of them land so that the
  next chunk opens a `CHAPTER` heading — a proxy for "the cut respected the
  document's structure".
* `find_midword_cut` finds the first fixed-split cut that tears a word in half
  (both sides of the cut end/start on an alphanumeric character) and
  reconstructs the torn word from the letter runs on both sides.
* `find_chapter_boundary` finds the first recursive-split cut where the next
  chunk opens a chapter heading, with the overlap repeat stripped from the tail
  so you see where the previous chunk truly ended.

`escape` makes newlines visible in the previews; `torn_word` and
`overlap_suffix` are the small string helpers behind the two finders.


In [7]:
def escape(s: str) -> str:
    """Make newlines visible so cut positions are obvious in the output."""
    return s.replace("\n", "\\n")


def torn_word(tail: str, head: str) -> str:
    """Reconstruct the word torn across a cut from the letter runs on both sides."""
    trailing = []
    for ch in reversed(tail):
        if ch.isalnum():
            trailing.append(ch)
        else:
            break
    leading = []
    for ch in head:
        if ch.isalnum():
            leading.append(ch)
        else:
            break
    return "".join(reversed(trailing)) + "".join(leading)


def overlap_suffix(prev: str, next_: str) -> str:
    """Longest suffix of ``prev`` that is also a prefix of ``next_`` (the overlap repeat)."""
    for n in range(min(len(prev), len(next_)), 0, -1):
        if prev[-n:] == next_[:n]:
            return prev[-n:]
    return ""


# Gutenberg novels are plain text: structure shows up as "CHAPTER 1"/"CHAPTER I"
# headings, not markdown "#" headings.
CHAPTER_RE = re.compile(r"(?i)^chapter\s+[0-9ivxlcdm.]+")


def cut_stats(chunks: list[Document]) -> tuple[int, int]:
    """Return (in-document cuts, cuts where the next chunk opens a chapter heading)."""
    total = 0
    chapter_aligned = 0
    for i in range(len(chunks) - 1):
        if chunks[i].metadata.get("source") != chunks[i + 1].metadata.get("source"):
            continue
        total += 1
        if CHAPTER_RE.match(chunks[i + 1].page_content.lstrip()):
            chapter_aligned += 1
    return total, chapter_aligned


def find_midword_cut(chunks: list[Document]) -> tuple[int, str, str, str] | None:
    """First in-document cut where fixed splitting tears a word in half.

    Returns (index, tail_preview, fresh_head_preview, torn_word). The next
    chunk's preview has the ``CHUNK_OVERLAP`` repeat stripped so the reader
    sees the continuation of the torn word, not the duplicated tail.
    """
    for i in range(len(chunks) - 1):
        if chunks[i].metadata.get("source") != chunks[i + 1].metadata.get("source"):
            continue
        tail = chunks[i].page_content
        head = chunks[i + 1].page_content
        overlap = tail[-CHUNK_OVERLAP:]
        fresh = head[len(overlap):] if head.startswith(overlap) else head
        if tail and fresh and tail[-1].isalnum() and fresh[0].isalnum():
            return i, tail[-PREVIEW:], fresh[:PREVIEW], torn_word(tail, fresh)
    return None


def find_chapter_boundary(chunks: list[Document]) -> tuple[int, str, str] | None:
    """First in-document cut where the next chunk opens a chapter heading.

    Returns (index, tail_preview, head_preview). The tail has the overlap
    repeat stripped so the reader sees where the previous chunk truly ended.
    """
    for i in range(len(chunks) - 1):
        if chunks[i].metadata.get("source") != chunks[i + 1].metadata.get("source"):
            continue
        head = chunks[i + 1].page_content
        if CHAPTER_RE.match(head.lstrip()):
            tail = chunks[i].page_content
            overlap = overlap_suffix(tail, head)
            if overlap:
                tail = tail[: -len(overlap)]
            return i, tail[-PREVIEW:], head[:PREVIEW]
    return None


# --- 5. Boundary quality ---------------------------------------------
f_cuts, f_aligned = cut_stats(fixed_chunks)
r_cuts, r_aligned = cut_stats(recursive_chunks)
print("\n[4] Where do the cuts land?")
print(f"    cuts landing at a chapter heading (next chunk opens a 'Chapter'): "
      f"fixed {f_aligned}/{f_cuts}, recursive {r_aligned}/{r_cuts}")

cut = find_midword_cut(fixed_chunks)
if cut is not None:
    i, tail, fresh, word = cut
    src = Path(fixed_chunks[i].metadata["source"]).name
    print(f"\n    FIXED cuts at exactly {CHUNK_SIZE} chars, mid-word:")
    print(f"      {src} chunk {i} ends    : ...{escape(tail)}")
    print(f"      {src} chunk {i + 1} (overlap stripped) starts: {escape(fresh)}...")
    print(f"      -> the word '{word}' is torn in half across chunks {i} and {i + 1}")
else:
    print("\n    FIXED: no mid-word cut found (text too short or boundaries aligned).")

boundary = find_chapter_boundary(recursive_chunks)
if boundary is not None:
    j, tail, head = boundary
    src = Path(recursive_chunks[j].metadata["source"]).name
    print("\n    RECURSIVE climbs the separator ladder and lands on a chapter boundary:")
    print(f"      {src} chunk {j} ends  : ...{escape(tail)}")
    print(f"      {src} chunk {j + 1} starts: {escape(head)}...")
    print(f"      -> cut lands at a paragraph boundary; chunk {j + 1} opens a clean chapter")
else:
    print("\n    RECURSIVE: no chapter boundary found (whole doc fits in one chunk).")



[4] Where do the cuts land?
    cuts landing at a chapter heading (next chunk opens a 'Chapter'): fixed 1/6010, recursive 137/7393

    FIXED cuts at exactly 500 chars, mid-word:
      pride-and-prejudice.txt chunk 2 ends    : ... J. Comyns Carr\n                      in acknowledgment of all I\n                       owe to his f
      pride-and-prejudice.txt chunk 3 (overlap stripped) starts: riendship and\n                    advice, these illustrations are\n                         gratefull...
      -> the word 'friendship' is torn in half across chunks 2 and 3

    RECURSIVE climbs the separator ladder and lands on a chapter boundary:
      pride-and-prejudice.txt chunk 91 ends  : ..._]]\n\n\n\n\n[Illustration:\n\n“I hope Mr. Bingley will like it”\n\n[_Copyright 1894 by George Allen._]]\n\n\n\n\n
      pride-and-prejudice.txt chunk 92 starts: CHAPTER II.\n\n\n[Illustration]\n\nMr. Bennet was among the earliest of those who waited on Mr. Bingley. ...
      -> cut lands at a parag

## 6 · Takeaway

Same budget, different philosophy: fixed splitting counts characters, recursive
splitting counts structure. The numbers and the cut previews above show the
consequence — recursive chunks keep words and paragraphs intact, so embeddings
see clean text at every chunk boundary.


In [8]:
print("\n[5] Takeaway")
print("    Fixed splitting counts characters; recursive splitting counts")
print("    structure. Same 500/50 budget, but recursive chunks keep words")
print("    and paragraphs intact, so embeddings see clean text at every")
print("    chunk boundary.")



[5] Takeaway
    Fixed splitting counts characters; recursive splitting counts
    structure. Same 500/50 budget, but recursive chunks keep words
    and paragraphs intact, so embeddings see clean text at every
    chunk boundary.
